In [ ]:
# === Google Colab: siapkan repo, dataset, dan modul bersama ===
# Cell ini otomatis mencari folder repo. Jika tidak ada (misal dibuka langsung
# dari link Colab GitHub), repo di-clone lalu working directory dipindah ke ML/uts.
import os
import subprocess
import sys
from pathlib import Path

REPO = "Aldiss1/PBL_SmartReceipt"
BRANCH = "main"


def _bootstrap():
    here = Path.cwd()
    for base in [here, *here.parents]:
        if (base / "ML" / "uts" / "pipeline_utils.py").exists():
            return base
    dest = Path("/content") / REPO.split("/")[-1]
    if not (dest / "ML" / "uts" / "pipeline_utils.py").exists():
        subprocess.run(
            [
                "git", "clone", "--depth", "1", "--branch", BRANCH,
                f"https://github.com/{REPO}.git", str(dest),
            ],
            check=True,
        )
    return dest


ROOT = _bootstrap()
os.chdir(ROOT / "ML" / "uts")
sys.path.insert(0, str(ROOT / "ML" / "uts"))
print("Repo  :", ROOT)
print("Kerja :", Path.cwd())


# UTS PBL Machine Learning — Perbandingan Model
## SmartReceipt Analyzer

Notebook ini menggabungkan hasil empat model yang dikerjakan masing-masing anggota:

| Anggota | Model |
|---|---|
| Rakagali | Multinomial Naive Bayes |
| Aldi | k-Nearest Neighbors (kNN) |
| Rijal | Support Vector Machine (Linear SVM) |
| Noval | Random Forest |

Semua model memakai **data, pra-pengolahan, pembagian data, dan TF-IDF yang sama**.


### 1. Muat Data & Ekstraksi Fitur

In [ ]:
import sys
from pathlib import Path

candidates = [Path.cwd(), *Path.cwd().parents]
for base in list(candidates):
    candidates += [base / "ML" / "uts", base / "uts"]
for base in candidates:
    if (base / "pipeline_utils.py").exists():
        sys.path.insert(0, str(base))
        break

import numpy as np
import pandas as pd
from IPython.display import display

import pipeline_utils as pu

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
print("Konfigurasi TF-IDF:", pu.TFIDF_CONFIG)


In [ ]:
train, val, test = pu.load_splits()

print(f"Data latih : {train.shape[0]:,} baris")
print(f"Data val   : {val.shape[0]:,} baris")
print(f"Data uji   : {test.shape[0]:,} baris")
print(f"Jumlah kategori target: {train['kategori'].nunique()}")


In [ ]:
vectorizer, x_train = pu.fit_features(train)
x_val = vectorizer.transform(val["text"])
x_test = vectorizer.transform(test["text"])

print(f"Bentuk matriks TF-IDF latih : {x_train.shape}")
print(f"Bentuk matriks TF-IDF uji   : {x_test.shape}")
print(f"Jumlah fitur (n-gram)       : {x_train.shape[1]:,}")
print(f"Kepadatan matriks           : {x_train.nnz / (x_train.shape[0] * x_train.shape[1]):.4%}")


### 2. Latih Semua Model

Konfigurasi hyperparameter mengikuti notebook masing-masing anggota.


In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

import time

models = {
    "Multinomial Naive Bayes": MultinomialNB(alpha=0.03, fit_prior=True),
    "k-Nearest Neighbors (kNN)": KNeighborsClassifier(n_neighbors=5, weights="distance", metric="cosine", n_jobs=-1),
    "Support Vector Machine (Linear SVM)": LinearSVC(C=1.0, class_weight="balanced", max_iter=5000, random_state=pu.SEED),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_features="sqrt",
        class_weight="balanced_subsample", n_jobs=-1, random_state=pu.SEED,
    ),
}

hasil_semua = []
for nama, model in models.items():
    mulai = time.time()
    model.fit(x_train, train["kategori"])
    hasil = pu.evaluate_model(nama, model, x_test, test["kategori"])
    hasil["Waktu Latih (detik)"] = round(time.time() - mulai, 2)
    hasil_semua.append(hasil)
    print(f"{nama:15} selesai dalam {hasil['Waktu Latih (detik)']} detik")


### 3. Tabel Perbandingan Performa

In [ ]:
tabel = pd.DataFrame([
    {k: v for k, v in hasil.items() if not k.startswith("_")}
    for hasil in hasil_semua
]).sort_values("Akurasi", ascending=False).reset_index(drop=True)

tabel.round(4)

### 4. Visual Perbandingan (Tabel Transpose)

In [ ]:
display(tabel.set_index("Model")[["Akurasi", "Presisi (macro)", "Recall (macro)", "F1-Score (macro)"]].T.round(4))

tabel.to_csv(pu.RESULTS_CSV, index=False)
print(f"Hasil perbandingan disimpan ke: {pu.RESULTS_CSV}")

### 5. Confusion Matrix Tiap Model

Confusion matrix membantu melihat kategori mana yang paling sering tertukar.


In [ ]:
for hasil in hasil_semua:
    print("=" * 80)
    print(hasil["Model"])
    print("=" * 80)
    display(hasil["_confusion"])

### 6. Pembahasan

- **kNN** memperoleh akurasi uji tertinggi, tetapi biaya prediksinya paling besar karena harus menghitung jarak ke seluruh data latih.
- **SVM** memiliki performa validasi terbaik dan margin yang tegas; hasil uji sedikit di bawah kNN.
- **Naive Bayes** paling cepat dilatih dan paling ringan diekspor ke aplikasi Flutter, dengan akurasi yang kompetitif.
- **Random Forest** menangkap pola non-linear, tetapi kurang cocok untuk vektor TF-IDF yang renggang dan berukuran besar.
- Kategori **Kesehatan**, **Minuman**, dan **Rumah Tangga** konsisten menjadi titik lemah karena kosakata antar kategori saling tumpang tindih.

**Rekomendasi:** Naive Bayes tetap dipilih untuk produksi (on-device, <5 ms/item),
sedangkan kNN/SVM dapat dipertimbangkan bila akurasi lebih diprioritaskan dan
inferensi dilakukan di server.

### 7. Kontribusi Tim

| Anggota | Peran | Kontribusi |
|---|---|---|
| Rakagali | Ketua / Model Naive Bayes | Menyusun pipeline bersama, melatih dan mengevaluasi Multinomial Naive Bayes, menyusun laporan |
| Aldi | Model k-Nearest Neighbors | Melatih dan mengevaluasi kNN, analisis pengaruh nilai k |
| Rijal | Model Support Vector Machine | Melatih dan mengevaluasi SVM, analisis hyperparameter C |
| Noval | Model Random Forest | Melatih dan mengevaluasi Random Forest, analisis jumlah pohon |
| Semua | Kolaborasi | Menyusun dataset, memverifikasi pembagian data, mereview hasil perbandingan model |
